# P7513 alignment overlay

This notebook recreates the `P7513_alignment_overlay` plot from the alignment QC stage using the pregenerated P7513 Xenium and MERSCOPE SpatialData zarrs.

The plotting cell below copies the alignment overlay logic into an editable notebook function, with publication-style controls for colors, point size, alpha, legend, and PDF saving.

## Kernel

Run this notebook with the MerXen environment that has `spatialdata` installed. On this machine, the workflow conda environment at `work/conda/env-5ab9b2fa7b005685690639bfde24c694` imports the plotting stack successfully.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib import patheffects as path_effects
import numpy as np
import spatialdata as sd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "merxen").exists():
    REPO_ROOT = Path("/home/becalia/code/MerXen")

src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from merxen.alignment.features import build_alignment_adata, shared_gene_subset
from merxen.plotting import save_figure

PLATFORM_COLORS = {
    "MERSCOPE": "#1f77b4",
    "XENIUM": "#d62728",
}


In [ ]:
PAIR_ID = "P7513"

MERSCOPE_ZARR_PATH = REPO_ROOT / "results" / PAIR_ID / "merscope" / "latest" / "latest_spatialdata.zarr"
XENIUM_ZARR_PATH = REPO_ROOT / "results" / PAIR_ID / "xenium" / "latest" / "latest_spatialdata.zarr"

OUTPUT_DIR = REPO_ROOT / "results" / PAIR_ID / "alignment_qc" / "alignment_qc_out"
OUTPUT_PATH = OUTPUT_DIR / f"{PAIR_ID}_alignment_overlay_interactive.png"
OUTPUT_PDF_PATH = OUTPUT_PATH.with_suffix(".pdf")

print(f"MERSCOPE: {MERSCOPE_ZARR_PATH}")
print(f"Xenium:   {XENIUM_ZARR_PATH}")
print(f"PNG output: {OUTPUT_PATH}")
print(f"PDF output: {OUTPUT_PDF_PATH}")


In [ ]:
xenium_sdata = sd.read_zarr(XENIUM_ZARR_PATH)
merscope_sdata = sd.read_zarr(MERSCOPE_ZARR_PATH)

fixed = build_alignment_adata(xenium_sdata, platform="XENIUM")
moving = build_alignment_adata(merscope_sdata, platform="MERSCOPE")
fixed, moving = shared_gene_subset(fixed, moving)

print(f"fixed Xenium cells: {fixed.n_obs:,}")
print(f"moving MERSCOPE cells: {moving.n_obs:,}")
print(f"shared genes: {fixed.n_vars:,}")
print("fixed source:", fixed.uns.get("merxen_alignment"))
print("moving source:", moving.uns.get("merxen_alignment"))


In [ ]:
def _add_interactive_scale_bar(
    ax: plt.Axes,
    *,
    length_um: float = 20.0,
    linewidth: float = 10.0,
    fontsize: float = 20.0,
) -> None:
    """Draw a prominent scale bar in the visual bottom-right corner."""
    x_left, x_right = ax.get_xlim()
    y_top, y_bottom = ax.get_ylim()
    x0, x1 = sorted((x_left, x_right))
    y0, y1 = sorted((y_top, y_bottom))
    x_span = x1 - x0
    y_span = y1 - y0
    if x_span <= 0 or y_span <= 0:
        return

    shown_length = min(length_um, x_span * 0.8)
    pad_x = x_span * 0.06
    pad_y = y_span * 0.12
    x_end = x1 - pad_x
    x_start = x_end - shown_length
    y = y1 - pad_y if y_top > y_bottom else y0 + pad_y
    text_y = y - y_span * 0.035 if y_top > y_bottom else y + y_span * 0.035
    text_effects = [
        path_effects.Stroke(linewidth=3.0, foreground="white"),
        path_effects.Normal(),
    ]
    line_effects = [
        path_effects.Stroke(linewidth=linewidth + 4.0, foreground="white"),
        path_effects.Normal(),
    ]
    ax.plot(
        [x_start, x_end],
        [y, y],
        color="black",
        linewidth=linewidth,
        solid_capstyle="butt",
        path_effects=line_effects,
        zorder=20,
    )
    ax.text(
        0.5 * (x_start + x_end),
        text_y,
        f"{shown_length:.0f} µm",
        color="black",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        fontweight="bold",
        path_effects=text_effects,
        zorder=20,
    )



def plot_alignment_overlay_interactive(
    fixed,
    moving,
    output_path: Path | str | None = None,
    *,
    xenium_color: str = PLATFORM_COLORS["XENIUM"],
    merscope_color: str = PLATFORM_COLORS["MERSCOPE"],
    point_size: float = 0.45,
    alpha: float = 0.24,
    figsize: tuple[float, float] = (8.0, 8.0),
    show_legend: bool = True,
    legend_fontsize: float = 10.0,
    legend_marker_size: float = 40.0,
    legend_loc: str = "best",
    scale_bar_um: float | None = 20.0,
    scale_bar_linewidth: float = 10.0,
    scale_bar_fontsize: float = 20.0,
    dpi: int = 220,
) -> plt.Figure:
    """Interactive copy of `plot_alignment_overlay` with publication styling."""
    fixed_xy = np.asarray(fixed.obsm["spatial"], dtype=float)
    moving_xy = np.asarray(moving.obsm["spatial"], dtype=float)

    fig, ax = plt.subplots(figsize=figsize)
    if len(moving_xy):
        ax.scatter(
            moving_xy[:, 0],
            moving_xy[:, 1],
            s=point_size,
            alpha=alpha,
            color=merscope_color,
            label="MERSCOPE",
            rasterized=True,
            zorder=1,
        )
    if len(fixed_xy):
        ax.scatter(
            fixed_xy[:, 0],
            fixed_xy[:, 1],
            s=point_size,
            alpha=alpha,
            color=xenium_color,
            label="Xenium",
            rasterized=True,
            zorder=2,
        )

    ax.set_aspect("equal", adjustable="box")
    ax.invert_yaxis()

    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_frame_on(False)

    if scale_bar_um is not None:
        _add_interactive_scale_bar(
            ax,
            length_um=scale_bar_um,
            linewidth=scale_bar_linewidth,
            fontsize=scale_bar_fontsize,
        )

    if show_legend:
        legend = ax.legend(loc=legend_loc, frameon=False, fontsize=legend_fontsize)
        legend_handles = getattr(legend, "legend_handles", None)
        if legend_handles is None:
            legend_handles = legend.legendHandles
        for handle in legend_handles:
            if hasattr(handle, "set_alpha"):
                handle.set_alpha(1.0)
            if hasattr(handle, "set_sizes"):
                handle.set_sizes([legend_marker_size])

    fig.tight_layout(pad=0.0)
    if output_path is not None:
        save_figure(fig, output_path, dpi=dpi)
    return fig


In [ ]:
# Optional: save the interactive output.
# save_figure() is the repo helper used by production plots; it writes PNG plus a sibling PDF.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig = plot_alignment_overlay_interactive(
    fixed,
    moving,
    output_path=OUTPUT_PATH,
    xenium_color=PLATFORM_COLORS["XENIUM"],
    merscope_color=PLATFORM_COLORS["MERSCOPE"],
    point_size=0.4,
    alpha=0.15,
    figsize=(8.0, 8.0),
    show_legend=True,
    legend_fontsize=20,
    legend_marker_size=120,
    legend_loc="best",
    scale_bar_um=2000,
    scale_bar_linewidth=10,
    scale_bar_fontsize=25,
)
print(f"Saved PNG: {OUTPUT_PATH}")
print(f"Saved PDF: {OUTPUT_PDF_PATH}")
